[Reference](https://python.plainenglish.io/i-thought-i-knew-python-then-these-7-features-humbled-me-completely-8a2c6b5862ce$0)

# 1. The Walrus Operator Does More Than You Think

In [1]:
results = []
for user_id in user_ids:
    data = fetch_user(user_id)      # expensive call
    if data and data["active"]:
        results.append(data)

In [2]:
results = [
    data
    for user_id in user_ids
    if (data := fetch_user(user_id)) and data["active"]
]

In [3]:
import re

lines = ["Error: disk full", "Info: started", "Error: timeout"]

errors = [
    match.group(1)
    for line in lines
    if (match := re.search(r"Error: (.+)", line))
]
# ['disk full', 'timeout']

# 2. Structural Pattern Matching Is Not a Switch Statement


In [4]:
def process_event(event: dict):
    match event:
        case {"type": "click", "button": "left", "x": x, "y": y}:
            handle_left_click(x, y)
        case {"type": "click", "button": "right"}:
            handle_right_click()
        case {"type": "keypress", "key": str(k)} if k.isupper():
            handle_uppercase_key(k)
        case {"type": "keypress", "key": k}:
            handle_key(k)
        case _:
            log_unknown_event(event)

In [5]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

def describe(shape):
    match shape:
        case Point(x=0, y=0):
            return "origin"
        case Point(x=0, y=y):
            return f"on y-axis at {y}"
        case Point(x=x, y=0):
            return f"on x-axis at {x}"
        case Point(x=x, y=y):
            return f"point at ({x}, {y})"

# 3. ```__slots__``` Can Cut Your Memory Usage in Half


In [6]:
# Without __slots__
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

In [7]:
# With __slots__
class PointSlotted:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

In [9]:
import sys

p1 = Point(1, 2)
p2 = PointSlotted(1, 2)

print(sys.getsizeof(p1.__dict__))   # 232 bytes
print(sys.getsizeof(p2))            # 56 bytes

288
48


# 4. functools.cache Is Not the Same as functools.lru_cache


In [10]:
from functools import cache, lru_cache

@lru_cache(maxsize=None)
def fib_lru(n):
    if n < 2:
        return n
    return fib_lru(n - 1) + fib_lru(n - 2)

@cache
def fib_cache(n):
    if n < 2:
        return n
    return fib_cache(n - 1) + fib_cache(n - 2)

In [11]:
from functools import cached_property

class DataProcessor:
    def __init__(self, filepath):
        self.filepath = filepath

    @cached_property
    def data(self):
        print("Loading data...")      # runs only once
        return load_large_file(self.filepath)

p = DataProcessor("data.csv")
_ = p.data    # loads file
_ = p.data    # returns cached result, no reload

# 5. Context Managers Are Not Just for File Handling


In [12]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label: str):
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"{label}: {elapsed:.4f}s")

In [13]:
with timer("database query"):
    results = db.execute("SELECT * FROM users")

In [14]:
import os
# What most people write
try:
    os.remove("temp.txt")
except FileNotFoundError:
    pass

In [15]:
# What contextlib gives you
from contextlib import suppress

with suppress(FileNotFoundError):
    os.remove("temp.txt")

# 6. Generator Pipelines Can Replace Entire Data Processing Chains


In [16]:
import csv

def read_rows(filepath):
    with open(filepath) as f:
        yield from csv.DictReader(f)

def filter_active(rows):
    return (row for row in rows if row["status"] == "active")

def parse_amount(rows):
    return (
        {**row, "amount": float(row["amount"])}
        for row in rows
    )

def above_threshold(rows, threshold):
    return (row for row in rows if row["amount"] > threshold)

# Compose the pipeline
rows = read_rows("transactions.csv")
rows = filter_active(rows)
rows = parse_amount(rows)
rows = above_threshold(rows, 1000.0)

for row in rows:
    process(row)

# 7. dataclasses Have Features Nobody Talks About


In [17]:
from dataclasses import dataclass, field
from typing import ClassVar

@dataclass
class Order:
    id: int
    items: list = field(default_factory=list)      # mutable default, done right
    _total: float = field(default=0.0, repr=False) # excluded from __repr__
    tax_rate: ClassVar[float] = 0.08               # class-level, not per-instance

    def add_item(self, price: float):
        self.items.append(price)
        self._total = sum(self.items) * (1 + self.tax_rate)

In [18]:
@dataclass
class PositiveInt:
    value: int

    def __post_init__(self):
        if self.value <= 0:
            raise ValueError(f"value must be positive, got {self.value}")

In [19]:
@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(1.0, 2.0)
cache = {p: "some result"}   # works because Point is now hashable